## 0. Install Dependencies

In [1]:
!pip install ultralytics -q
!pip install -U ultralytics -q  # ensure latest version

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Ultralytics version: 8.3.241
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB


## 1. Configuration

In [2]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
OBB_ROOT       = r'D:\leaf.v1-yolo_leaf1.yolov5-obb'
CLASSIFIER_PATH = r'D:\Leaf dataset 2\outputs\CustomCNN\CustomCNN_best.keras'
OUTPUT_DIR     = os.path.join(OBB_ROOT, 'outputs')
DATA_YAML      = os.path.join(OBB_ROOT, 'data.yaml')

# ── Classes ────────────────────────────────────────────────────────────────
OBB_CLASSES = [
    'Blight_a', 'Blight_b', 'Blight_c',
    'Danger_a', 'Danger_b',
    'Healthy_a', 'Healthy_b'
]
CLASS_MAP = {name: idx for idx, name in enumerate(OBB_CLASSES)}

CLASSIFIER_CLASSES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust",
    "Apple___healthy", "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot",
    "Peach___healthy", "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot",
    "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

SEVERITY_MAP = {
    'Healthy_a': '🟢 Healthy (Confirmed)',
    'Healthy_b': '🟡 Healthy (Minor marks)',
    'Danger_a' : '🟠 Warning — Early Stage',
    'Danger_b' : '🟠 Caution — Moderate',
    'Blight_a' : '🔴 Blight — Mild',
    'Blight_b' : '🔴 Blight — Moderate',
    'Blight_c' : '🔴 Blight — SEVERE ⚠️',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Configuration loaded")
print(f"   OBB classes  : {OBB_CLASSES}")
print(f"   Classifier   : {len(CLASSIFIER_CLASSES)} classes")
print(f"   Output dir   : {OUTPUT_DIR}")

✅ Configuration loaded
   OBB classes  : ['Blight_a', 'Blight_b', 'Blight_c', 'Danger_a', 'Danger_b', 'Healthy_a', 'Healthy_b']
   Classifier   : 38 classes
   Output dir   : D:\leaf.v1-yolo_leaf1.yolov5-obb\outputs


## 2. Convert DOTA Labels → YOLOv8-OBB Format

DOTA format: `x1 y1 x2 y2 x3 y3 x4 y4 ClassName Difficult` (absolute pixels)  
YOLOv8-OBB format: `class_idx x1 y1 x2 y2 x3 y3 x4 y4` (normalized 0-1)

In [ ]:
import glob
from PIL import Image
from tqdm import tqdm

def convert_dota_to_yoloobb(label_dir, image_dir, output_dir, class_map):
    os.makedirs(output_dir, exist_ok=True)
    label_files = glob.glob(os.path.join(label_dir, '*.txt'))
    converted, skipped, empty = 0, 0, 0

    for lf in tqdm(label_files, desc=f'Converting {os.path.basename(label_dir)}'):
        filename = os.path.basename(lf)

        # Find matching image (try common extensions)
        img_path = None
        stem = os.path.splitext(filename)[0]
        for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
            candidate = os.path.join(image_dir, stem + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            # Write empty label so YOLOv8 knows it's a background image
            open(os.path.join(output_dir, filename), 'w').close()
            continue

        img = Image.open(img_path)
        W, H = img.size

        out_lines = []
        with open(lf, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 9:
                    continue

                try:
                    x1, y1 = float(parts[0]), float(parts[1])
                    x2, y2 = float(parts[2]), float(parts[3])
                    x3, y3 = float(parts[4]), float(parts[5])
                    x4, y4 = float(parts[6]), float(parts[7])
                    class_name = parts[8]
                except ValueError:
                    continue

                if class_name not in class_map:
                    continue

                class_idx = class_map[class_name]

                # Clamp and normalize
                def norm_x(v): return max(0.0, min(1.0, v / W))
                def norm_y(v): return max(0.0, min(1.0, v / H))

                out_lines.append(
                    f"{class_idx} {norm_x(x1):.6f} {norm_y(y1):.6f} "
                    f"{norm_x(x2):.6f} {norm_y(y2):.6f} "
                    f"{norm_x(x3):.6f} {norm_y(y3):.6f} "
                    f"{norm_x(x4):.6f} {norm_y(y4):.6f}"
                )

        out_path = os.path.join(output_dir, filename)
        with open(out_path, 'w') as f:
            f.write('\n'.join(out_lines))

        if len(out_lines) == 0:
            empty += 1
        converted += 1

    print(f"  ✅ Converted : {converted} | ⚠️ No image found : {skipped} | 📄 Empty (background) : {empty}")


# Run conversion for all splits
for split in ['train', 'valid', 'test']:
    print(f"\n🔄 Processing '{split}'...")
    convert_dota_to_yoloobb(
        label_dir  = os.path.join(OBB_ROOT, split, 'labelTxt'),
        image_dir  = os.path.join(OBB_ROOT, split, 'images'),
        output_dir = os.path.join(OBB_ROOT, split, 'labels'),
        class_map  = CLASS_MAP,
    )

print("\n✅ Label conversion complete!")


🔄 Processing 'train'...


Converting labelTxt:  89%|████████▉ | 10534/11822 [01:18<00:08, 145.43it/s]

## 3. Verify Labels

In [ ]:
print("=" * 60)
print("  Label Verification")
print("=" * 60)

for split in ['train', 'valid', 'test']:
    label_files = glob.glob(os.path.join(OBB_ROOT, split, 'labels', '*.txt'))
    non_empty   = sum(1 for f in label_files if os.path.getsize(f) > 0)
    total_boxes = 0
    class_counts = {i: 0 for i in range(len(OBB_CLASSES))}

    for lf in label_files:
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 9:
                    total_boxes += 1
                    class_counts[int(parts[0])] += 1

    print(f"\n📁 {split}:")
    print(f"   Label files : {len(label_files)}")
    print(f"   Non-empty   : {non_empty}")
    print(f"   Total boxes : {total_boxes}")
    print(f"   Per class   :")
    for idx, name in enumerate(OBB_CLASSES):
        bar = '█' * int(class_counts[idx] / max(class_counts.values()) * 20)
        print(f"     {name:<12} {class_counts[idx]:>5}  {bar}")

    # Preview sample
    sample = next((f for f in label_files if os.path.getsize(f) > 0), None)
    if sample:
        with open(sample) as f:
            line = f.readline().strip()
        print(f"   Sample line : {line}")

## 4. Create data.yaml

In [ ]:
yaml_content = f"""# YOLOv8-OBB Plant Disease Dataset
path: {OBB_ROOT}
train: train/images
val: valid/images
test: test/images

nc: {len(OBB_CLASSES)}
names: {OBB_CLASSES}
"""

with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created:")
print("-" * 40)
print(yaml_content)

## 5. Train YOLOv8m-OBB

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m-obb.pt')  # pretrained backbone

results = model.train(
    data     = DATA_YAML,
    epochs   = 100,
    imgsz    = 640,
    batch    = 16,
    device   = 0,
    project  = OUTPUT_DIR,
    name     = 'yolov8m_obb_v2',
    patience = 20,           # early stopping

    # Optimizer
    optimizer = 'AdamW',
    lr0       = 0.001,
    cos_lr    = True,
    weight_decay = 0.0005,

    # Loss weights — boost cls to help with imbalanced Healthy classes
    cls = 1.0,
    box = 7.5,
    dfl = 1.5,

    # Augmentation
    fliplr     = 0.5,
    flipud     = 0.2,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    mosaic     = 1.0,
    mixup      = 0.1,       # helps rare Healthy_a/b classes
    copy_paste = 0.1,       # copies rare instances into other images

    # Misc
    plots   = True,
    verbose = True,
    amp     = True,         # mixed precision for RTX 4060
)

## 6. Evaluate on Test Set

In [ ]:
import os
import glob

# Find best weights
weight_pattern = os.path.join(OUTPUT_DIR, 'yolov8m_obb_v2', 'weights', 'best.pt')
best_weights = weight_pattern

if not os.path.exists(best_weights):
    # fallback: find any best.pt
    candidates = glob.glob(os.path.join(OUTPUT_DIR, '**', 'best.pt'), recursive=True)
    best_weights = candidates[-1] if candidates else None

print(f"📍 Best weights: {best_weights}")

# Load trained model and evaluate
trained_model = YOLO(best_weights)

test_metrics = trained_model.val(
    data   = DATA_YAML,
    split  = 'test',
    imgsz  = 640,
    device = 0,
)

print("\n" + "=" * 50)
print("  Test Set Results")
print("=" * 50)
print(f"  mAP50      : {test_metrics.box.map50:.4f}")
print(f"  mAP50-95   : {test_metrics.box.map:.4f}")
print(f"  Precision  : {test_metrics.box.mp:.4f}")
print(f"  Recall     : {test_metrics.box.mr:.4f}")

## 7. Per-Class Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Per-class AP from validation
val_metrics = trained_model.val(
    data   = DATA_YAML,
    split  = 'val',
    imgsz  = 640,
    device = 0,
    verbose = True,
)

# Build results dataframe
rows = []
for i, name in enumerate(OBB_CLASSES):
    rows.append({
        'Class'   : name,
        'AP50'    : float(val_metrics.box.ap50[i]) if i < len(val_metrics.box.ap50) else 0.0,
        'AP50-95' : float(val_metrics.box.ap[i])  if i < len(val_metrics.box.ap)   else 0.0,
    })

df = pd.DataFrame(rows).sort_values('AP50', ascending=False)
print(df.to_string(index=False))

# Bar chart
colors = ['#e74c3c' if 'Blight' in c else '#e67e22' if 'Danger' in c else '#2ecc71'
          for c in df['Class']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df['Class'], df['AP50'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('AP@50', fontsize=12)
ax.set_title('Per-Class AP@50 — YOLOv8m-OBB', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.axhline(y=val_metrics.box.map50, color='navy', linestyle='--', alpha=0.7,
           label=f'mAP50 = {val_metrics.box.map50:.3f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='Blight'),
    mpatches.Patch(color='#e67e22', label='Danger'),
    mpatches.Patch(color='#2ecc71', label='Healthy'),
]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color='navy',
          linestyle='--', label=f'mAP50 = {val_metrics.box.map50:.3f}')])

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'per_class_ap50.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Plot saved → {plot_path}")

## 8. Two-Stage Inference Pipeline

**Stage 1:** YOLOv8-OBB detects lesion regions + assigns severity grade  
**Stage 2:** CustomCNN classifier identifies specific disease from cropped region  
**Output:** Disease name + severity level + confidence scores

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgba

# ── Load models ──────────────────────────────────────────────────────────
print("Loading OBB model...")
obb_model = YOLO(best_weights)

print("Loading classifier...")
classifier = tf.keras.models.load_model(CLASSIFIER_PATH)
print("✅ Both models loaded")

# ── Severity colour map for visualisation ────────────────────────────────
SEVERITY_COLORS = {
    'Healthy_a': (0,   200, 0),    # green
    'Healthy_b': (0,   180, 80),   # green-ish
    'Danger_a' : (255, 165, 0),    # orange
    'Danger_b' : (255, 120, 0),    # dark orange
    'Blight_a' : (220, 50,  50),   # red
    'Blight_b' : (180, 20,  20),   # dark red
    'Blight_c' : (130, 0,   0),    # very dark red
}

def extract_crop(image, obb_box, padding=10):
    """Extract axis-aligned crop from an OBB detection."""
    pts = obb_box.xyxyxyxy[0].cpu().numpy().astype(int)
    x1 = max(0, pts[:, 0].min() - padding)
    y1 = max(0, pts[:, 1].min() - padding)
    x2 = min(image.shape[1], pts[:, 0].max() + padding)
    y2 = min(image.shape[0], pts[:, 1].max() + padding)
    return image[y1:y2, x1:x2], (x1, y1, x2, y2)


def classify_crop(crop_bgr):
    """Run CustomCNN classifier on a BGR crop."""
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_resized = cv2.resize(crop_rgb, (224, 224))
    inp = np.expand_dims(crop_resized.astype('float32') / 255.0, axis=0)
    preds = classifier.predict(inp, verbose=0)
    idx = np.argmax(preds[0])
    return CLASSIFIER_CLASSES[idx], float(preds[0][idx])


def predict_image(image_path, conf_threshold=0.3, show=True, save_path=None):
    """
    Full two-stage pipeline on a single image.
    Returns list of detection dicts.
    """
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Stage 1 — OBB detection
    obb_results = obb_model(image_path, conf=conf_threshold, verbose=False)[0]

    detections = []

    if obb_results.obb is None or len(obb_results.obb) == 0:
        print("⚠️  No lesions detected. Image may be healthy or conf_threshold too high.")
        return detections

    for i, box in enumerate(obb_results.obb):
        severity_idx  = int(box.cls[0])
        severity_name = OBB_CLASSES[severity_idx]
        obb_conf      = float(box.conf[0])

        crop, (x1, y1, x2, y2) = extract_crop(image, box)
        if crop.size == 0:
            continue

        # Stage 2 — classification
        disease_name, cls_conf = classify_crop(crop)

        det = {
            'region'        : i + 1,
            'disease'       : disease_name.replace('___', ' — '),
            'severity'      : SEVERITY_MAP[severity_name],
            'severity_key'  : severity_name,
            'det_conf'      : obb_conf,
            'cls_conf'      : cls_conf,
            'bbox'          : (x1, y1, x2, y2),
        }
        detections.append(det)

    # Visualisation
    if show or save_path:
        img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(img_rgb)

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            color = tuple(c/255 for c in SEVERITY_COLORS[det['severity_key']])
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor=(*color, 0.08)
            )
            ax.add_patch(rect)

            label = (
                f"#{det['region']} {det['disease'].split(' — ')[1]}\n"
                f"{det['severity'].split(' ')[1]} "
                f"(det:{det['det_conf']:.0%} cls:{det['cls_conf']:.0%})"
            )
            ax.text(x1, y1 - 5, label, color='white', fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

        ax.set_title(f"Detections: {len(detections)} | {os.path.basename(image_path)}",
                     fontsize=12, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"💾 Saved → {save_path}")
        if show:
            plt.show()
        plt.close()

    # Print summary
    print(f"\n{'─'*65}")
    print(f"  📍 Image : {os.path.basename(image_path)}")
    print(f"  🔍 Detections : {len(detections)}")
    print(f"{'─'*65}")
    for det in detections:
        print(f"  Region {det['region']:>2}: {det['disease']:<45}")
        print(f"           {det['severity']}")
        print(f"           Det conf: {det['det_conf']:.2%}  |  Cls conf: {det['cls_conf']:.2%}")
        print()

    return detections

print("✅ Pipeline function ready — use predict_image(path) to run inference")

## 9. Run Inference on Test Images

In [ ]:
# ── Run on a few test images ─────────────────────────────────────────────
test_images = glob.glob(os.path.join(OBB_ROOT, 'test', 'images', '*.jpg'))[:5]
test_images += glob.glob(os.path.join(OBB_ROOT, 'test', 'images', '*.JPG'))[:5]
test_images += glob.glob(os.path.join(OBB_ROOT, 'test', 'images', '*.png'))[:5]
test_images = list(set(test_images))[:5]  # up to 5 unique images

if not test_images:
    print("⚠️  No test images found. Check path.")
else:
    for img_path in test_images:
        save_path = os.path.join(
            OUTPUT_DIR,
            'inference_' + os.path.basename(img_path).replace(' ', '_')
        )
        predict_image(img_path, conf_threshold=0.3, show=True, save_path=save_path)

## 10. Inference on a Custom Image
Change the path below to test on any image.

In [ ]:
# ── Change this path to your image ───────────────────────────────────────
CUSTOM_IMAGE = r'D:\path\to\your\leaf_image.jpg'

if os.path.exists(CUSTOM_IMAGE):
    results = predict_image(
        CUSTOM_IMAGE,
        conf_threshold = 0.25,   # lower = more detections, higher = more precise
        show           = True,
        save_path      = os.path.join(OUTPUT_DIR, 'custom_prediction.png'),
    )
else:
    print(f"⚠️  File not found: {CUSTOM_IMAGE}")
    print("    Update CUSTOM_IMAGE path above and re-run.")

## 11. Batch Inference & Export Results to CSV

In [ ]:
import pandas as pd
from datetime import datetime

def batch_predict(image_dir, conf_threshold=0.3, output_csv=None):
    """Run pipeline on all images in a directory and save results to CSV."""
    exts  = ['*.jpg', '*.JPG', '*.jpeg', '*.png', '*.PNG']
    paths = []
    for ext in exts:
        paths += glob.glob(os.path.join(image_dir, ext))
    paths = list(set(paths))

    print(f"Found {len(paths)} images in {image_dir}")

    all_rows = []
    for img_path in tqdm(paths, desc='Batch inference'):
        try:
            dets = predict_image(img_path, conf_threshold=conf_threshold, show=False)
            for det in dets:
                all_rows.append({
                    'image'         : os.path.basename(img_path),
                    'region'        : det['region'],
                    'disease'       : det['disease'],
                    'severity'      : det['severity'],
                    'det_confidence': round(det['det_conf'], 4),
                    'cls_confidence': round(det['cls_conf'], 4),
                })
            if not dets:
                all_rows.append({
                    'image'    : os.path.basename(img_path),
                    'region'   : 0,
                    'disease'  : 'No detection',
                    'severity' : '—',
                    'det_confidence': 0,
                    'cls_confidence': 0,
                })
        except Exception as e:
            print(f"  ⚠️  Error on {os.path.basename(img_path)}: {e}")

    df = pd.DataFrame(all_rows)

    if output_csv is None:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_csv = os.path.join(OUTPUT_DIR, f'batch_predictions_{ts}.csv')

    df.to_csv(output_csv, index=False)
    print(f"\n✅ Saved {len(df)} rows → {output_csv}")

    # Summary
    print("\n📊 Severity distribution:")
    print(df['severity'].value_counts().to_string())

    return df


# Example usage — run on test set
test_dir = os.path.join(OBB_ROOT, 'test', 'images')
df_results = batch_predict(test_dir, conf_threshold=0.3)

## 12. Summary

In [ ]:
print("=" * 60)
print("  🌿 Plant Disease Detection — Final Summary")
print("=" * 60)
print()
print("  Stage 1 — YOLOv8m-OBB Detector")
print(f"    Classes  : {OBB_CLASSES}")
print(f"    Weights  : {best_weights}")
print(f"    mAP50    : see cell 6 output")
print()
print("  Stage 2 — CustomCNN Classifier")
print(f"    Classes  : {len(CLASSIFIER_CLASSES)} disease classes")
print(f"    Val acc  : 99.28%")
print(f"    Weights  : {CLASSIFIER_PATH}")
print()
print("  Pipeline Output")
print("    'Tomato — Late_blight | 🔴 Blight — SEVERE ⚠️ | Det:87% Cls:94%'")
print()
print("  Outputs saved to:")
print(f"    {OUTPUT_DIR}")
print("=" * 60)